# Notebook 06 — AlphaFold2 and the Protein Folding Revolution

**Module:** 11 — Structural Biology  
**Tier:** 3 — Survey  
**Notebook:** 06 of 08  
**Time estimate:** 75 minutes

> In 2021, AlphaFold2 achieved atomic-accuracy structure prediction for most proteins.
> This is not hyperbole — CASP14 results showed predictions within experimental error.
> Understanding what AlphaFold2 does (and doesn't do) is now mandatory knowledge
> for any computational biologist.

---
## Step 1 — Motivation

Every bioinformatics interview in 2024–2027 will involve AlphaFold2. The question
is not whether you've heard of it — everyone has — but whether you can explain:
1. What the protein folding problem was
2. What AlphaFold2 does architecturally
3. How to interpret its output (pLDDT, PAE)
4. What it cannot do (dynamics, mutations, interactions, disordered regions)

---
## Step 2 — Intuition

**Before AlphaFold2:**  
Structure prediction methods fell into:
- Homology modelling: copy from a known structure of a similar sequence (~60%+ identity)
- Threading/fold recognition: match to known folds
- De novo (Rosetta): energy minimization; worked only for small proteins

For proteins without a close template, predictions were often wrong.

**AlphaFold2's key insight:**  
Use the co-evolutionary signal in a Multiple Sequence Alignment (MSA) of homologous
sequences to infer spatial constraints (like contact maps, but richer), process these
through a deep Transformer architecture (Evoformer), then directly predict 3D atomic
coordinates through a structure module that respects geometric constraints.

**Levinthal's paradox (1969):**  
A 100-residue protein has ~100^100 possible conformations. Even sampling 10^13
conformations/second, exhaustive search would take longer than the universe's age.
Yet proteins fold in microseconds. AlphaFold2 solved Levinthal's paradox not by
searching conformational space, but by directly predicting the answer from sequence.

---
## Step 3 — Biological Background

### CASP (Critical Assessment of Protein Structure Prediction)

Blind prediction competition run every two years since 1994. Organizers collect
unpublished experimental structures; predictors submit models before the structures
are released. CASP14 (2020): AlphaFold2 achieved median GDT_TS of 92.4 — within
the range of experimental variation. Declared the problem "solved" for single
chains in well-folded regions.

### AlphaFold2 Architecture (Jumper et al. 2021)

**Input:**
1. **MSA (Multiple Sequence Alignment):** Homologous sequences searched from Uniclust30,
   BFD, MGnify. Co-evolutionary signal is in the MSA — positions that co-vary.
2. **Pair representation:** Initial pair features (distance bins, templates if available)

**Evoformer (48 blocks):**  
Jointly processes the MSA representation and the pair representation:
- MSA row-wise/column-wise attention: updates MSA representation
- Triangle multiplication/attention: updates pair representation (respecting triangle
  inequality geometry)
- MSA → pair: outer product mean (MSA updates pairs)
- Pair → MSA: pair bias in attention

**Structure Module (8 blocks):**  
- Takes the pair representation and builds 3D coordinates
- Uses **SE(3)-equivariant** frames (invariant point attention, IPA)
- Each residue is represented as a rigid body (N, Cα, C backbone frame)
- Iteratively refines backbone, then side-chain torsion angles

**Recycling:** The process is iterated 3–4 times, feeding predictions back in.

### pLDDT and PAE

**pLDDT (predicted Local Distance Difference Test):**  
Per-residue confidence score, 0–100. Predicts what the LDDT score would be if the
structure were experimentally determined. Interpretation:
- pLDDT > 90: very high confidence (experimental accuracy)
- pLDDT 70–90: confident, likely correct backbone
- pLDDT 50–70: low confidence, use with caution
- pLDDT < 50: predicted disordered/unstructured region

**PAE (Predicted Aligned Error):**  
Matrix of expected position error (Å) for each pair of residues. Low PAE = confident
relative positions. Used to identify domain arrangements in multimers.

---
## Step 4 — Mathematical Explanation

**LDDT (Local Distance Difference Test) score:**  
For a predicted structure vs. reference:
$$\text{LDDT} = \frac{1}{|L|} \sum_{l \in \text{thresholds}} \frac{|\{(i,j): |d_{ij}^{\text{pred}} - d_{ij}^{\text{ref}}| < l\}|}{|\{(i,j): d_{ij}^{\text{ref}} < 15 \text{ Å}\}|}$$

Thresholds $l \in \{0.5, 1, 2, 4\}$ Å. LDDT = fraction of local distances within threshold.
**pLDDT** is AlphaFold2's prediction of this score before comparing to the true structure.

**Triangle inequality in pair representation:**  
If residues $i$, $j$, $k$ are in 3D space:
$$|d_{ij} - d_{jk}| \leq d_{ik} \leq d_{ij} + d_{jk}$$

AlphaFold2's triangle multiplication updates enforce this constraint: the pair
representation at $(i, k)$ is updated using information from $(i, j)$ and $(j, k)$
for all $j$ — similar to iterative refinement of a distance matrix.

In [ ]:
# Step 6 — Python: Simulate pLDDT profiles and interpret AlphaFold2 output
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

rng = np.random.default_rng(42)

# ---- Simulate a realistic pLDDT profile ----
# Protein: 300 residues
# N-terminal structured domain (residues 1–120): high pLDDT
# Linker (121–150): disordered, low pLDDT
# C-terminal structured domain (151–260): high pLDDT
# Disordered tail (261–300): very low pLDDT
N = 300
plddt = np.zeros(N)
# Structured N-terminal domain
plddt[:120] = np.clip(rng.normal(88, 5, 120), 60, 100)
# Disordered linker
plddt[120:150] = np.clip(rng.normal(42, 10, 30), 20, 65)
# Structured C-terminal domain
plddt[150:260] = np.clip(rng.normal(82, 8, 110), 55, 100)
# Disordered C-terminal tail
plddt[260:300] = np.clip(rng.normal(30, 12, 40), 15, 55)

print('pLDDT profile summary:')
print(f'  Residues 1-120 (structured): mean={plddt[:120].mean():.1f}')
print(f'  Residues 121-150 (linker):   mean={plddt[120:150].mean():.1f}')
print(f'  Residues 151-260 (structured): mean={plddt[150:260].mean():.1f}')
print(f'  Residues 261-300 (tail):    mean={plddt[260:].mean():.1f}')
print(f'  Overall mean pLDDT: {plddt.mean():.1f}')
print(f'  Fraction >90: {(plddt > 90).mean()*100:.0f}%')
print(f'  Fraction 70-90: {((plddt >= 70) & (plddt <= 90)).mean()*100:.0f}%')
print(f'  Fraction <50 (disordered): {(plddt < 50).mean()*100:.0f}%')

# ---- Simulate CASP benchmark history ----
casp_years  = [1994, 1996, 1998, 2000, 2002, 2004, 2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020]
casp_gdt_best = [20, 25, 30, 35, 40, 45, 52, 55, 60, 65, 68, 72, 75, 92]
casp_gdt_median = [10, 12, 15, 18, 22, 28, 35, 38, 42, 48, 52, 55, 58, 73]

# ---- Simulate PAE matrix (predicted aligned error) ----
# Low PAE within domains, high PAE between domains due to flexible linker
PAE = np.full((N, N), 15.0)
# Within domain 1
PAE[:120, :120] = rng.exponential(2.5, (120, 120))
PAE[:120, :120] = (PAE[:120, :120] + PAE[:120, :120].T) / 2
np.fill_diagonal(PAE[:120, :120], 0)
# Within domain 2
PAE[150:260, 150:260] = rng.exponential(3.0, (110, 110))
PAE[150:260, 150:260] = (PAE[150:260, 150:260] + PAE[150:260, 150:260].T) / 2
np.fill_diagonal(PAE[150:260, 150:260], 0)
PAE = np.clip(PAE, 0, 30)
print(f'\nPAE matrix simulated: {PAE.shape}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: pLDDT profile
ax = axes[0, 0]
residues = np.arange(1, N + 1)
colors_plddt = []
for p in plddt:
    if p > 90:
        colors_plddt.append('royalblue')
    elif p >= 70:
        colors_plddt.append('cyan')
    elif p >= 50:
        colors_plddt.append('orange')
    else:
        colors_plddt.append('red')
ax.bar(residues, plddt, color=colors_plddt, width=1.0, edgecolor='none')
ax.axhline(90, color='royalblue', ls='--', lw=0.8, alpha=0.7)
ax.axhline(70, color='cyan',     ls='--', lw=0.8, alpha=0.7)
ax.axhline(50, color='orange',   ls='--', lw=0.8, alpha=0.7)
ax.set_xlabel('Residue')
ax.set_ylabel('pLDDT')
ax.set_title('A. Simulated pLDDT profile')
ax.set_xlim(1, N)
ax.set_ylim(0, 100)
legend_patches = [
    Patch(color='royalblue', label='>90: Very high'),
    Patch(color='cyan',      label='70–90: Confident'),
    Patch(color='orange',    label='50–70: Low'),
    Patch(color='red',       label='<50: Disordered'),
]
ax.legend(handles=legend_patches, fontsize=7, loc='lower right')

# Panel B: CASP benchmark — GDT_TS over time
ax = axes[0, 1]
ax.plot(casp_years[:-1], casp_gdt_best[:-1], 'o-', color='steelblue', label='Best group', lw=2)
ax.plot(casp_years[:-1], casp_gdt_median[:-1], 's--', color='grey', label='Median', lw=1.5)
ax.scatter([2020], [92], color='red', s=100, zorder=5, label='AlphaFold2 (CASP14)')
ax.scatter([2020], [73], color='orange', s=70, zorder=5)
ax.annotate('AlphaFold2\nrevolution', xy=(2020, 92), xytext=(2015, 85),
              arrowprops={'arrowstyle': '->', 'color': 'red'}, color='red', fontsize=8)
ax.set_xlabel('CASP year')
ax.set_ylabel('GDT_TS score')
ax.set_title('B. CASP benchmark history\n(higher = better prediction)')
ax.legend(fontsize=8)
ax.set_ylim(0, 100)

# Panel C: PAE matrix
ax = axes[1, 0]
im = ax.imshow(PAE, cmap='Greens_r', aspect='auto', vmin=0, vmax=30)
plt.colorbar(im, ax=ax, label='Predicted Aligned Error (Å)')
ax.set_xlabel('Residue')
ax.set_ylabel('Residue')
ax.set_title('C. Simulated PAE matrix\n(dark = confident relative positions)')
ax.axhline(120, color='white', ls='--', lw=0.8)
ax.axvline(120, color='white', ls='--', lw=0.8)
ax.axhline(150, color='white', ls='--', lw=0.8)
ax.axvline(150, color='white', ls='--', lw=0.8)

# Panel D: AlphaFold2 architecture overview (text)
ax = axes[1, 1]
ax.axis('off')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('D. AlphaFold2 Architecture Overview')
steps = [
    ('Input', 'Sequence + MSA (homologs) + Templates', 0.90),
    ('Embedding', 'MSA representation + Pair representation', 0.78),
    ('Evoformer ×48', 'Row/col attention on MSA; Triangle mult. on pairs', 0.66),
    ('Structure Module ×8', 'IPA: iterative backbone + torsion angle refinement', 0.52),
    ('Output', '3D coords + pLDDT per residue + PAE matrix', 0.40),
    ('Recycling ×3', 'Feed output back as input for iterative refinement', 0.28),
]
for name, desc, y in steps:
    ax.text(0.02, y, f'[{name}]', fontsize=9, fontweight='bold', color='steelblue')
    ax.text(0.02, y - 0.07, f'  {desc}', fontsize=7.5, color='black')
    if y > 0.30:
        ax.annotate('', xy=(0.08, y - 0.12), xytext=(0.08, y - 0.08),
                      arrowprops={'arrowstyle': '->', 'color': 'grey'})

plt.suptitle('Module 11 NB06: AlphaFold2 and the Protein Folding Revolution',
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('alphafold2.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Interpret the simulated pLDDT profile. Which regions would you trust for
   docking a small molecule? Which would you discard?
2. What is Levinthal's paradox and how does AlphaFold2 bypass it?
3. AlphaFold2 uses an MSA as input. What happens to prediction quality if a
   protein has very few homologous sequences (shallow MSA)?
4. (Survey) Look up AlphaFold-Multimer. What additional problem does it solve
   that the original AlphaFold2 cannot?

---
## Step 10 — Quiz

1. What is pLDDT and what does it measure? Interpret: pLDDT = 45 for a region.
2. What are the two main components of AlphaFold2's architecture?
3. What is CASP and why is CASP14 historically significant?
4. Name three things AlphaFold2 cannot predict reliably.
5. What is the PAE matrix and when is it more informative than pLDDT?

---
## Step 12 — Reflection

> *[In one paragraph: a colleague says "now that AlphaFold2 exists, we don't need
> experimental structure determination anymore." Explain two scenarios where
> experimental structures are still essential and AlphaFold2 predictions are
> insufficient.]*

---
*Next: `07_structure_comparison_and_drug_design.ipynb`*